### RUN IN TPU ~01:30:00 to process 1998-2026




In [ ]:
# CELL 0: Set Run Parameters
import os

os.environ["CACHE_LOOKBACK"] = "41"
os.environ["CACHE_START_DATE"] = "1998-01-01"

In [ ]:
# CELL 1
import time

start_time = time.time()

In [ ]:
# CELL 2: Setup
import sys, os
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2"
    )
else:
    rl_root = Path("C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

In [ ]:
# CELL 3: Load Data & Initialize Cache
from core.settings import TradingConfig, CacheConfig
from core.paths import OUTPUT_DIR
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar
from data_pipeline.screener import UniverseScreener
from data_pipeline.cache import CheckpointAlphaCache  # Extracted from original notebook

print("Loading preprocessed data...")
df_ohlcv, macro_df, features_df = load_processed_data()
config = TradingConfig()

# Reconstruct wide df_close dynamically and sync calendar
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

In [ ]:
# CELL 4: Build Cache
import os
from pathlib import Path
from core.paths import OUTPUT_DIR, LOCAL_DATA_DIR  # Added LOCAL_DATA_DIR

# Create directories if they do not exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Successfully ensured output directory exists: {OUTPUT_DIR}")
print(f"Successfully ensured local data directory exists: {LOCAL_DATA_DIR}")

In [ ]:
# CELL 5: Build Cache
cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()

# Add a warning if the cache file already exists
if cache_file_path.exists():
    print(
        f"Warning: Cache file already exists at {cache_file_path}. Resuming build process."
    )

cache = CheckpointAlphaCache(
    screener=screener, config=config, lookbacks=[CacheConfig.LOOKBACK]
)

cache.build(
    start_date=CacheConfig.START_DATE,
    end_date=None,
    checkpoint_path=cache_file_path,
    save_every_n_days=2000,  # save every ~7 minutes
)

# print(cache.describe())

In [ ]:
# CELL 6: Timing
end_time = time.time()
print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(end_time - start_time))}")

In [ ]:
# CELL 7: Disconnect from TPU
from google.colab import runtime

# This will restart and unassign the current runtime,
# effectively disconnecting from the T4 GPU.
runtime.unassign()
print("Runtime unassigned. This session has been disconnected.")